In [0]:
# =============================================================================
# setup/governance_setup.py
# Manually triggered — NOT part of the automated Bronze/Silver/Gold pipeline
# or CI/CD deploy. Grants and policies change when the RBAC design changes,
# not on every commit. Run once per environment, on demand.
# =============================================================================

dbutils.widgets.text("target_catalog", "prd_mwua_capstone_team2")
dbutils.widgets.dropdown("environment", "prod", ["dev", "prod"])

catalog = dbutils.widgets.get("target_catalog")
environment = dbutils.widgets.get("environment")

spark.sql(f"USE CATALOG {catalog}")
print(f"Applying governance setup to {catalog} ({environment})")


# =============================================================================
# grp_team2_data_engineers — full access in dev, read-only in prod
# =============================================================================
if environment == "dev":
    for schema in ["landing", "bronze", "silver", "gold", "reference"]:
        spark.sql(f"GRANT ALL PRIVILEGES ON SCHEMA {catalog}.{schema} TO grp_team2_data_engineers")
else:
    for schema in ["bronze", "silver", "gold", "reference"]:
        spark.sql(f"GRANT SELECT ON SCHEMA {catalog}.{schema} TO grp_team2_data_engineers")


# =============================================================================
# grp_team2_data_platform — governance seat + dim_zone approver
# =============================================================================
spark.sql(f"GRANT SELECT ON SCHEMA {catalog}.silver TO grp_team2_data_platform")
spark.sql(f"GRANT SELECT ON SCHEMA {catalog}.gold TO grp_team2_data_platform")
spark.sql(f"GRANT SELECT ON SCHEMA {catalog}.reference TO grp_team2_data_platform")
spark.sql(f"GRANT APPLY TAG ON SCHEMA {catalog}.silver TO grp_team2_data_platform")
spark.sql(f"GRANT APPLY TAG ON SCHEMA {catalog}.gold TO grp_team2_data_platform")
spark.sql(f"GRANT MODIFY ON TABLE {catalog}.reference.dim_zone TO grp_team2_data_platform")


# =============================================================================
# grp_team2_executive — all Gold, cross-domain scorecard
# =============================================================================
spark.sql(f"GRANT SELECT ON SCHEMA {catalog}.gold TO grp_team2_executive")
spark.sql(f"GRANT SELECT ON TABLE {catalog}.reference.dim_zone TO grp_team2_executive")


# =============================================================================
# grp_team2_operations — billing + network Gold
# =============================================================================
for tbl in ["gold.billing_by_zone_month",
            "gold.network_health_by_zone_month",
            "gold.network_health_diagnostic_by_zone_month"]:
    spark.sql(f"GRANT SELECT ON TABLE {catalog}.{tbl} TO grp_team2_operations")
spark.sql(f"GRANT SELECT ON TABLE {catalog}.reference.dim_zone TO grp_team2_operations")


# =============================================================================
# grp_team2_finance_contractor_ops — finance + contractor domain
# =============================================================================
for tbl in ["silver.finance_invoices", "silver.invoice_line_items", "silver.works_orders",
            "gold.spend_by_zone_month", "gold.contractor_by_zone_month"]:
    spark.sql(f"GRANT SELECT ON TABLE {catalog}.{tbl} TO grp_team2_finance_contractor_ops")
spark.sql(f"GRANT SELECT ON TABLE {catalog}.reference.dim_zone TO grp_team2_finance_contractor_ops")


# =============================================================================
# grp_team2_billing_pii_unmask — table access (masking behavior set up below)
# =============================================================================
for tbl in ["silver.billing_consumption", "silver.customer_pii", "gold.billing_by_zone_month"]:
    spark.sql(f"GRANT SELECT ON TABLE {catalog}.{tbl} TO grp_team2_billing_pii_unmask")

print("Table/schema grants complete.")


# =============================================================================
# Governed tags — must exist BEFORE tagging columns or creating the policy.
# Account-level objects, not scoped to one catalog. Requires account or
# workspace admin permission to create.
# =============================================================================
try:
    spark.sql("CREATE GOVERNED TAG sensitivity VALUES ('pii', 'internal', 'public')")
    print("Governed tag 'sensitivity' created.")
except Exception as e:
    if "already exists" in str(e).lower() or "ALREADY_EXISTS" in str(e):
        print("Governed tag 'sensitivity' already exists — continuing.")
    else:
        raise

try:
    spark.sql("CREATE GOVERNED TAG domain VALUES ('billing', 'finance', 'network')")
    print("Governed tag 'domain' created.")
except Exception as e:
    if "already exists" in str(e).lower() or "ALREADY_EXISTS" in str(e):
        print("Governed tag 'domain' already exists — continuing.")
    else:
        raise


# =============================================================================
# Tag the PII columns
# =============================================================================
spark.sql(f"ALTER TABLE {catalog}.silver.customer_pii ALTER COLUMN customer_name SET TAGS ('sensitivity' = 'pii', 'domain' = 'billing')")
spark.sql(f"ALTER TABLE {catalog}.silver.customer_pii ALTER COLUMN address SET TAGS ('sensitivity' = 'pii', 'domain' = 'billing')")
spark.sql(f"ALTER TABLE {catalog}.silver.customer_pii ALTER COLUMN contact_number SET TAGS ('sensitivity' = 'pii', 'domain' = 'billing')")
print("PII columns tagged.")


# =============================================================================
# Masking function
# =============================================================================
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.silver.mask_pii(val STRING)
RETURNS STRING
RETURN CASE
    WHEN is_account_group_member('grp_team2_billing_pii_unmask') THEN val
    ELSE '***MASKED***'
END
""")
print("Masking function created.")


# =============================================================================
# ABAC policy — drop-if-exists via try/except (DROP POLICY has no IF EXISTS
# clause), has_tag_value() for exact value matching, ON COLUMN required for
# column mask policies.
# =============================================================================
try:
    spark.sql(f"DROP POLICY mask_billing_pii ON CATALOG {catalog}")
    print("Existing policy dropped.")
except Exception as e:
    if "POLICY_NOT_FOUND" in str(e):
        print("No existing policy to drop — first run, continuing.")
    else:
        raise

spark.sql(f"""
CREATE POLICY mask_billing_pii
ON CATALOG {catalog}
COLUMN MASK {catalog}.silver.mask_pii
TO `account users`
EXCEPT `grp_team2_billing_pii_unmask`
FOR TABLES
MATCH COLUMNS has_tag_value('sensitivity', 'pii') AND has_tag_value('domain', 'billing') AS value
ON COLUMN value
""")
print("Policy created.")

print(f"\nGovernance setup complete for {catalog} ({environment})")


# =============================================================================
# Verification — run manually, as different users, after the script completes
# =============================================================================
# Not in grp_team2_billing_pii_unmask: should return ***MASKED***
# SELECT customer_name, address, contact_number FROM {catalog}.silver.customer_pii LIMIT 5;

# In grp_team2_billing_pii_unmask: should return real values
# SELECT customer_name, address, contact_number FROM {catalog}.silver.customer_pii LIMIT 5;